# Preamble

In [ ]:
import matplotlib.pylab as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import re

In [ ]:
def increment_number_in_string(input_string):
    pattern = r'\d+'
    match = re.search(pattern, input_string)

    if match:
        number_str = match.group(0)
        incremented_number = str(int(number_str) + 1)

        result_string = re.sub(pattern, incremented_number, input_string, count=1)
        return result_string
    else:
        return input_string

In [ ]:
SIZE = 25
plt.rcParams['axes.labelsize']  = SIZE
plt.rcParams['legend.fontsize'] = SIZE
plt.rcParams['xtick.labelsize'] = SIZE
plt.rcParams['ytick.labelsize'] = SIZE
plt.rcParams['font.size']       = SIZE

In [ ]:
OCEAN_TYPES = ['atlantic' , 'western_pacific', 'central_pacific', 'eastern_pacific', 'indian_ocean']
SEASONS = ['MAM', 'JJA', 'SON', 'DJF']
YEARS = ['2020', '2021', '2022']

rows = []

for ocean_type in OCEAN_TYPES:
    for year in YEARS:
        for season in SEASONS:
            if season == 'DJF':
                year = increment_number_in_string(year)
            else:
                year = year
            if season == 'MAM' and year == '2022':
                break
            print(ocean_type, year, season)

            ZM_uu_wind_icon_orig = np.load(f'00_data/07_fig_08_data/{ocean_type}/08_fig_ZM_uu_wind_icon_orig_{ocean_type}_{year}_{season}.npy')
            ZM_uu_RFM = np.load(f'00_data/07_fig_08_data/{ocean_type}/08_fig_ZM_uu_RFM_{ocean_type}_{year}_{season}.npy')
            ZM_uu_wind_revised = np.load(f'00_data/07_fig_08_data/{ocean_type}/08_fig_ZM_uu_wind_revised_{ocean_type}_{year}_{season}.npy')

            ZM_vv_wind_icon_orig = np.load(f'00_data/07_fig_08_data/{ocean_type}/08_fig_ZM_vv_wind_icon_orig_{ocean_type}_{year}_{season}.npy')
            ZM_vv_RFM = np.load(f'00_data/07_fig_08_data/{ocean_type}/08_fig_ZM_vv_RFM_{ocean_type}_{year}_{season}.npy')
            ZM_vv_wind_revised = np.load(f'00_data/07_fig_08_data/{ocean_type}/08_fig_ZM_vv_wind_revised_{ocean_type}_{year}_{season}.npy')

            # Compute RMSE for zonal winds
            control_array = ZM_uu_wind_icon_orig
            array1 = ZM_uu_RFM
            array2 = ZM_uu_wind_revised

            if control_array.shape == array1.shape == array2.shape:
                rmse_uu_icon_vs_RFM = np.sqrt(np.mean((control_array - array1) ** 2))
                rmse_uu_icon_vs_WIN = np.sqrt(np.mean((control_array - array2) ** 2))
            else:
                print("Error: Zonal wind arrays do not have the same shape. Cannot compute RMSE.")
                rmse_uu_icon_vs_RFM = rmse_uu_icon_vs_WIN = np.nan

            # Compute RMSE for meridional winds
            control_array = ZM_vv_wind_icon_orig
            array1 = ZM_vv_RFM
            array2 = ZM_vv_wind_revised

            if control_array.shape == array1.shape == array2.shape:
                rmse_vv_icon_vs_RFM = np.sqrt(np.mean((control_array - array1) ** 2))
                rmse_vv_icon_vs_WIN = np.sqrt(np.mean((control_array - array2) ** 2))
            else:
                print("Error: Meridional wind arrays do not have the same shape. Cannot compute RMSE.")
                rmse_vv_icon_vs_RFM = rmse_vv_icon_vs_WIN = np.nan

            # Append results to list
            rows.append({
                'Ocean_Type': ocean_type,
                'Year': year,
                'Season': season,
                'rmse_uu_icon_vs_RFM': rmse_uu_icon_vs_RFM,
                'rmse_uu_icon_vs_WIN': rmse_uu_icon_vs_WIN,
                'rmse_vv_icon_vs_RFM': rmse_vv_icon_vs_RFM,
                'rmse_vv_icon_vs_WIN': rmse_vv_icon_vs_WIN
            })

# Convert the list of rows to a DataFrame
RMSE_values = pd.DataFrame(rows)
RMSE_values

In [ ]:
wind_pattern = pd.read_csv(f'what_wind_pattern.csv', index_col=0)
wind_pattern

In [ ]:
wind_pattern['Year'] = wind_pattern['Year'].astype(str)
RMSE_values['Year'] = RMSE_values['Year'].astype(str)

selected_values = []

for _, row in wind_pattern.iterrows():
    ocean_type = row['Ocean_Type']
    year = row['Year']
    season = row['Season']
    wind_pattern_type = row['Wind Pattern']

    rmse_row = RMSE_values[
        (RMSE_values['Ocean_Type'] == ocean_type) &
        (RMSE_values['Year'] == year) &
        (RMSE_values['Season'] == season)
    ]

    if not rmse_row.empty:
        if wind_pattern_type == "Zonal Wind Pattern":
            selected_values.append({
                "Ocean_Type": ocean_type,
                "Year": year,
                "Season": season,
                "Wind Pattern": wind_pattern_type,
                "RMSE_RFM": rmse_row.iloc[0]["rmse_uu_icon_vs_RFM"],
                "RMSE_WIN": rmse_row.iloc[0]["rmse_uu_icon_vs_WIN"]
            })
        elif wind_pattern_type == "Meridional Wind Pattern":
            selected_values.append({
                "Ocean_Type": ocean_type,
                "Year": year,
                "Season": season,
                "Wind Pattern": wind_pattern_type,
                "RMSE_RFM": rmse_row.iloc[0]["rmse_vv_icon_vs_RFM"],
                "RMSE_WIN": rmse_row.iloc[0]["rmse_vv_icon_vs_WIN"]
            })

SELECTED_values = pd.DataFrame(selected_values)
SELECTED_values

In [ ]:
ocean_order = ["atlantic", "indian_ocean", "western_pacific", "central_pacific", "eastern_pacific"]

season_order = ["MAM", "JJA", "SON", "DJF"]

SELECTED_values['Ocean_Type_order'] = SELECTED_values['Ocean_Type'].apply(lambda x: ocean_order.index(x))

SELECTED_values['Season_order'] = SELECTED_values['Season'].apply(lambda x: season_order.index(x))

SELECTED_values = SELECTED_values.sort_values(
    by=["Ocean_Type_order", "Year", "Season_order"],
    ascending=[True, True, True]
)

SELECTED_values = SELECTED_values.drop(columns=["Ocean_Type_order", "Season_order"])
SELECTED_values

### Sorted by Wind Pattern

In [ ]:
wPcf_zonal_wind_pattern_ICON_vs_RFM = SELECTED_values.loc[
    (SELECTED_values['Ocean_Type'] == 'western_pacific'),
    'RMSE_RFM'
].values 

wPcf_zonal_wind_pattern_ICON_vs_WIN = SELECTED_values.loc[
    (SELECTED_values['Ocean_Type'] == 'western_pacific'),
    'RMSE_WIN'
].values 

In [ ]:
cPcf_zonal_wind_pattern_ICON_vs_RFM = SELECTED_values.loc[
    (SELECTED_values['Ocean_Type'] == 'central_pacific'),
    'RMSE_RFM'
].values 

cPcf_zonal_wind_pattern_ICON_vs_WIN = SELECTED_values.loc[
    (SELECTED_values['Ocean_Type'] == 'central_pacific'),
    'RMSE_WIN'
].values 

In [ ]:
IO_zonal_wind_pattern_ICON_vs_RFM = SELECTED_values.loc[
    (SELECTED_values['Ocean_Type'] == 'indian_ocean'),
    'RMSE_RFM'
].values 

IO_zonal_wind_pattern_ICON_vs_WIN = SELECTED_values.loc[
    (SELECTED_values['Ocean_Type'] == 'indian_ocean'),
    'RMSE_WIN'
].values 

In [ ]:
ePcf_meridional_wind_pattern_ICON_vs_RFM = SELECTED_values.loc[
    (SELECTED_values['Ocean_Type'] == 'eastern_pacific'),
    'RMSE_RFM'
].values 

ePcf_meridional_wind_pattern_ICON_vs_WIN = SELECTED_values.loc[
    (SELECTED_values['Ocean_Type'] == 'eastern_pacific'),
    'RMSE_WIN'
].values 

In [ ]:
ATL_meridional_wind_pattern_ICON_vs_RFM = SELECTED_values.loc[
    (SELECTED_values['Ocean_Type'] == 'atlantic'),
    'RMSE_RFM'
].values 

ATL_meridional_wind_pattern_ICON_vs_WIN = SELECTED_values.loc[
    (SELECTED_values['Ocean_Type'] == 'atlantic'),
    'RMSE_WIN'
].values 

# Plotting

In [ ]:
SIZE = 20
plt.rcParams['axes.labelsize']  = SIZE
plt.rcParams['legend.fontsize'] = SIZE
plt.rcParams['xtick.labelsize'] = SIZE
plt.rcParams['ytick.labelsize'] = SIZE
plt.rcParams['font.size']       = SIZE

In [ ]:
fig = plt.figure(figsize=(14,20), facecolor='w', edgecolor='k')
G = gridspec.GridSpec(5,1, hspace=0.8)

ax1 = plt.subplot(G[0,0])

ax1.plot(wPcf_zonal_wind_pattern_ICON_vs_WIN, marker='o', ms=15, lw=3, color='#FFC914', label='Western Pacific', clip_on=False, zorder=12)
ax1.plot(wPcf_zonal_wind_pattern_ICON_vs_RFM, marker='x', ls='dashed', ms=15, markeredgewidth=5, lw=3, color='#FFC914', label='Western Pacific', clip_on=False, zorder=12)

ax1.set_title('Western Pacific', pad=10)
ax1.spines[['top','right']].set_visible(False)
ax1.spines[['bottom', 'left']].set_position(('outward',20))
ax1.set_xlim(0,7)
ax1.set_xticks(range(8))
ax1.set_xticklabels(["MAM-'20", "JJA-'20", "SON-'20", "DJF-'20", "MAM-'21", "JJA-'21", "SON-'21", "DJF-'21"])
ax1.set_ylim(bottom=0)
#ax1.set_yticks([0,500,1000,1500,2000,2500])
#ax1.set_xlabel('Season')
ax1.set_ylabel(r'RMSE / $ms^{-1}$')

ax2 = plt.subplot(G[1,0])

ax2.plot(cPcf_zonal_wind_pattern_ICON_vs_WIN, marker='o', ms=15, lw=3, color='#2E282A', label='ICON vs. Revised', clip_on=False, zorder=14)
ax2.plot(cPcf_zonal_wind_pattern_ICON_vs_RFM, marker='x', ls='dashed', ms=15, markeredgewidth=5, lw=3, color='#2E282A', label='ICON vs. RFM', clip_on=False, zorder=14)

ax2.set_title('Central Pacific', pad=10)
ax2.spines[['top','right']].set_visible(False)
ax2.spines[['bottom', 'left']].set_position(('outward',20))
ax2.set_xlim(0,7)
ax2.set_xticks(range(8))
ax2.set_xticklabels(["MAM-'20", "JJA-'20", "SON-'20", "DJF-'20", "MAM-'21", "JJA-'21", "SON-'21", "DJF-'21"])
ax2.set_ylim(bottom=0)
#ax2.set_yticks([0,500,1000,1500,2000,2500])
#ax2.set_xlabel('Season')
ax2.set_ylabel(r'RMSE / $ms^{-1}$')

ax1 = plt.subplot(G[2,0])

ax1.plot(ePcf_meridional_wind_pattern_ICON_vs_WIN, marker='o', ms=15, lw=3, color='#76B041', label='Eastern Pacific', clip_on=False)
ax1.plot(ePcf_meridional_wind_pattern_ICON_vs_RFM, marker='x', ls='dashed', ms=15, markeredgewidth=5, lw=3, color='#76B041', label='Eastern Pacific', clip_on=False)

ax1.set_title('Eastern Pacific', pad=10)
ax1.spines[['top','right']].set_visible(False)
ax1.spines[['bottom', 'left']].set_position(('outward',20))
ax1.set_xlim(0,7)
ax1.set_xticks(range(8))
ax1.set_xticklabels(["MAM-'20", "JJA-'20", "SON-'20", "DJF-'20", "MAM-'21", "JJA-'21", "SON-'21", "DJF-'21"])
ax1.set_ylim(bottom=0)
#ax1.set_yticks([0,500,1000,1500,2000,2500])
#ax1.set_xlabel('Season')
ax1.set_ylabel(r'RMSE / $ms^{-1}$')


ax1 = plt.subplot(G[3,0])

ax1.plot(ATL_meridional_wind_pattern_ICON_vs_WIN, marker='o', ms=15, lw=3, color='#E4572E', label='Atlantic', clip_on=False)
ax1.plot(ATL_meridional_wind_pattern_ICON_vs_RFM, marker='x', ls='dashed', ms=15, markeredgewidth=5, lw=3, color='#E4572E', label='Atlantic', clip_on=False)

ax1.set_title('Atlantic', pad=10)
ax1.spines[['top','right']].set_visible(False)
ax1.spines[['bottom', 'left']].set_position(('outward',20))
ax1.set_xlim(0,7)
ax1.set_xticks(range(8))
ax1.set_xticklabels(["MAM-'20", "JJA-'20", "SON-'20", "DJF-'20", "MAM-'21", "JJA-'21", "SON-'21", "DJF-'21"])
ax1.set_ylim(bottom=0)
#ax1.set_yticks([0,500,1000,1500,2000,2500])
#ax1.set_xlabel('Season')
ax1.set_ylabel(r'RMSE / $ms^{-1}$')

ax1 = plt.subplot(G[4,0])

ax1.plot(IO_zonal_wind_pattern_ICON_vs_WIN, marker='o', ms=15, lw=3, color='#17BEBB', label='Indian Ocean', zorder=10, clip_on=False)
ax1.plot(IO_zonal_wind_pattern_ICON_vs_RFM, marker='x', ls='dashed', ms=15, markeredgewidth=5, lw=3, color='#17BEBB', label='Indian Ocean', zorder=10, clip_on=False)

ax1.set_title('Indian Ocean', pad=10)
ax1.spines[['top','right']].set_visible(False)
ax1.spines[['bottom', 'left']].set_position(('outward',20))
ax1.set_xlim(0,7)
ax1.set_xticks(range(8))
ax1.set_xticklabels(["MAM-'20", "JJA-'20", "SON-'20", "DJF-'20", "MAM-'21", "JJA-'21", "SON-'21", "DJF-'21"])
ax1.set_ylim(bottom=0)
#ax1.set_yticks([0,500,1000,1500,2000,2500])
ax1.set_xlabel('Season')
ax1.set_ylabel(r'RMSE / $ms^{-1}$')

ax2.legend(loc='upper center', bbox_to_anchor=(0.51,-6), fancybox=True, shadow=False, ncol=5)

plt.tight_layout()

filename = f'fig_S2.png'
filepath = '01_figs/'
plt.savefig(filepath + filename, facecolor='white', bbox_inches='tight', dpi=400)

plt.show()